In [1]:
import os
from dotenv import load_dotenv
_ = load_dotenv()

from langchain_postgres import PGEngine, PGVectorStore
from sqlalchemy.ext.asyncio import create_async_engine
from langchain_openai import OpenAIEmbeddings

In [2]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
pg_engine = PGEngine.from_connection_string(url= os.environ["POSTGRES_URL_2"])

In [3]:
# Set an existing table name
TABLE_NAME = "CanonicalNames"
# SCHEMA_NAME = "my_schema"

# Initialize PGVectorStore
name_store = await PGVectorStore.create(
    engine=pg_engine,
    table_name=TABLE_NAME,
    # schema_name=SCHEMA_NAME,
    embedding_service=embeddings,
    # Connect to existing VectorStore by customizing below column names
    id_column="id",
    content_column="raw_name",
    embedding_column="embedding",
    metadata_columns=["name_type", "canonical_name", "website_url"],
)

In [ ]:
import uuid

from langchain_core.documents import Document


events = [{'event_name': 'Some History History',
  'date': 'April 1, 2025 - April 1, 2026 (recurring monthly on the 1st)',
  'time': '8:30 PM to 9:30 PM',
  'source_url': 'https://www.jacksonholehistory.org',
  'event_main_url': 'https://www.jacksonholehistory.org',
  'event_secondary_url': 'https://www.jacksonholehistory.org',
  'event_address': '240 S Glenwood Street, Suite 124, Jackson, WY 83001',
  'venue_name': 'Jackson Hole History Museum',
  'host_organization_name': 'History Jackson Hole',
  'event_description': 'Some Description.'},
  {'event_name': 'Some History History',
  'date': 'April 1, 2025 - April 1, 2026 (recurring monthly on the 1st)',
  'time': '8:30 PM to 9:30 PM',
  'source_url': 'https://www.jacksonholehistory.org',
  'event_main_url': 'https://www.jacksonholehistory.org',
  'event_secondary_url': 'https://www.jacksonholehistory.org',
  'event_address': '240 S Glenwood Street, Suite 124, Jackson, WY 83001',
  'venue_name': 'History Jackson Hole',
  'host_organization_name': 'Jackshon Hole Historical Society and Museum',
  'event_description': 'Something else'}
 ]

name_fields = [
    "host_organization_name",
    "venue_name",
    "event_name",
]

docs = []
seen = set()

for event in events:
    for field in name_fields:
        value = event.get(field)
        if not value or not str(value).strip():
            continue

        composite_name = str(value).strip()
        key = (field, composite_name.lower())

        if key in seen:
            continue
        seen.add(key)

        docs.append(
            Document(
                id=str(uuid.uuid4()),
                page_content=composite_name,  # -> content_column="composite_name"
                metadata={
                    "name_type": field,
                    "canonical_name": composite_name,
                },
            )
        )


In [ ]:
print(docs)

In [ ]:

await name_store.aadd_documents(docs)

In [ ]:
from langchain_postgres.v2.indexes import HNSWIndex

index = HNSWIndex(name="my-hnsw-index")
await name_store.aapply_vector_index(index)

In [ ]:
filter={"name_type": {"$eq": "venue_name"}}

In [ ]:
query = "The coach"
query_vector = embeddings.embed_query(query)
docs = await name_store.asimilarity_search_by_vector(query_vector, k=2, filter=filter)
## print doc metadata field 'canonical_name'
for doc in docs:
    print(doc.metadata['canonical_name'])

In [ ]:
query = "JH Outdoor Leadership"
query_vector = embeddings.embed_query(query)
docs = await name_store.asimilarity_search_by_vector(query_vector, k=2)
print(docs)